In [2]:
!pip install pytesseract

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [12]:
import cv2
import pytesseract
import re
import os
import csv

# Update this path if Tesseract is not in your System Path (Windows users)
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def validate_kenyan_plate(text):
    """
    Validates the Kenyan format: KXX 123X (e.g., KDA 456B)
    """
    pattern = r'K[A-Z]{2}\s?\d{3}[A-Z]'
    text = text.strip().upper()
    if re.fullmatch(pattern, text):
        return True, text
    return False, text

def process_image(image_path):
    # 1. Load and Grayscale
    img = cv2.imread(image_path)
    if img is None:
        return None, "Error: Image not found"
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 2. Noise reduction and Edge detection
    # Applying Bilateral Filter to remove noise while keeping edges sharp
    filtered = cv2.bilateralFilter(gray, 11, 17, 17)
    edged = cv2.Canny(filtered, 30, 200)

    # 3. Find Contours (Looking for rectangular shapes)
    contours, _ = cv2.findContours(edged.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]

    screenCnt = None
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.018 * peri, True)
        if len(approx) == 4:  # Looking for a quadrilateral
            screenCnt = approx
            break

    if screenCnt is None:
        return None, "No Plate Detected"

    # 4. Masking and Cropping
    x, y, w, h = cv2.boundingRect(screenCnt)
    roi = gray[y:y+h, x:x+w]

    # 5. OCR Extraction
    # psm 7: Treat the image as a single text line.
    config = '--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    raw_text = pytesseract.image_to_string(roi, config=config)
    
    # Clean OCR noise
    clean_text = "".join(raw_text.split())
    
    return roi, clean_text

def run_pipeline(input_folder, output_csv):
    results = []
    
    if not os.path.exists(input_folder):
        print("Folder not found.")
        return

    for filename in os.listdir(input_folder):
        if filename.endswith(('.jpg', '.png', '.jpeg')):
            path = os.path.join(input_folder, filename)
            roi, text = process_image(path)
            
            is_valid, formatted_text = validate_kenyan_plate(text)
            status = "Valid Kenyan Plate" if is_valid else "Invalid"
            
            # Display result if plate found
            if roi is not None:
                cv2.imshow(f"Plate: {filename}", roi)
                print(f"File: {filename} | Extracted: {formatted_text} | Status: {status}")
                cv2.waitKey(1000) # Show for 1 second
            
            results.append([filename, formatted_text, status])

    # 6. Save to CSV
    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['filename', 'plate_number', 'validity'])
        writer.writerows(results)
    
    cv2.destroyAllWindows()
    print(f"\nProcessing complete. Logs saved to {output_csv}")

# EXECUTION
run_pipeline('car_images', 'results.csv')

File: images (2).png | Extracted:  | Status: Invalid

Processing complete. Logs saved to results.csv
